In [30]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [31]:
load_dotenv()  # Load environment variables from .env file

True

In [32]:
model = ChatOpenAI(model_name="gpt-4o-mini")

In [33]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description="Detailed feedback on the essay")
    score:int = Field(description="Score out of 10", ge=0, le=10)

In [34]:
structured_model = model.with_structured_output(EvaluationSchema)

In [35]:
essay = """# India in the Age of Artificial Intelligence

Artificial Intelligence, commonly known as AI, is one of the most transformative technologies of the modern era. From smartphones and online shopping to healthcare, education, agriculture, banking, and manufacturing, AI is rapidly becoming part of everyday life. For a country as large and diverse as India, the age of AI presents both an enormous opportunity and a significant responsibility. India has the potential to become one of the world's leading AI-powered economies if it combines technological innovation with education, responsible development, and inclusive growth.

India has several advantages in the AI revolution. The country has a large population of young people, a strong information-technology industry, a growing startup ecosystem, and a large pool of engineers and software professionals. Indian companies and startups are increasingly using AI to automate processes, analyze large amounts of data, improve customer services, and develop new products. The growth of AI is also creating opportunities for Indian businesses to compete in global markets.

One of the most important areas where AI can benefit India is education. AI-powered learning platforms can provide students with personalized learning experiences based on their individual strengths and weaknesses. Intelligent tutoring systems can explain difficult concepts, generate practice questions, and provide immediate feedback. Teachers can also use AI to reduce repetitive administrative work and focus more on teaching and interacting with students. However, AI should support teachers rather than completely replace human interaction and judgment.

Healthcare is another sector where AI can have a major impact. AI systems can assist doctors in analyzing medical images, identifying patterns in patient data, and supporting early detection of certain diseases. In rural and underserved areas, AI-enabled digital healthcare solutions could help improve access to medical information and services. At the same time, healthcare AI must be developed carefully because medical decisions involve privacy, accuracy, ethics, and human responsibility.

Agriculture, which remains an important part of India's economy, can also benefit from artificial intelligence. AI can analyze weather patterns, soil conditions, satellite images, and crop data to help farmers make better decisions. It can assist in identifying crop diseases, predicting yields, and optimizing the use of water and fertilizers. These technologies could increase productivity while helping farmers use resources more efficiently.

AI is also transforming India's manufacturing and industrial sectors. Intelligent machines can monitor equipment, detect faults, predict maintenance requirements, and improve production efficiency. In factories, AI can analyze machine data in real time and identify problems before they result in major breakdowns. This can reduce costs, improve safety, and increase productivity. However, workers will need to develop new skills to work alongside increasingly intelligent systems.

The financial sector is another major area of AI adoption. Banks and financial institutions can use AI for fraud detection, risk analysis, customer support, and financial forecasting. AI-powered systems can process huge volumes of transactions much faster than humans. This can make financial services more efficient, but it also increases the importance of cybersecurity and data protection.

Despite these opportunities, India's AI journey also presents serious challenges. One major concern is employment. Automation may reduce the need for humans to perform certain repetitive tasks. This does not necessarily mean that AI will eliminate all jobs; instead, the nature of many jobs may change. Workers will increasingly need skills such as programming, data analysis, problem-solving, communication, creativity, and the ability to work with AI systems. Therefore, reskilling and upskilling will become essential.

Another challenge is the digital divide. Not every Indian has equal access to high-speed internet, computers, modern educational resources, or digital services. If AI development benefits only technologically advanced cities and organizations, it could increase existing inequalities. India therefore needs to ensure that the benefits of AI reach rural communities, small businesses, students, farmers, and other sections of society.

Data privacy and responsible AI are equally important. AI systems depend heavily on data, and inappropriate collection or use of personal information can create serious risks. AI systems can also produce incorrect or biased results. India needs strong practices around privacy, cybersecurity, transparency, accountability, and responsible AI development.

The future of India in the age of AI will therefore depend not only on how advanced its technology becomes, but also on how wisely that technology is used. India needs to invest in AI research, computing infrastructure, education, and digital literacy while encouraging innovation and entrepreneurship. Schools and universities should prepare students not simply to use AI tools but to understand their limitations, risks, and ethical implications.

Ultimately, AI should be viewed as a tool for human progress rather than a replacement for human intelligence. India's greatest strength is not only its technology but also its people. If India's engineers, entrepreneurs, researchers, teachers, policymakers, and citizens work together, AI can become a powerful force for economic growth and social development.

In conclusion, India stands at an important point in its technological journey. The age of AI brings opportunities to improve education, healthcare, agriculture, manufacturing, finance, and public services, while also creating challenges related to employment, inequality, privacy, and ethics. The goal should not be to adopt AI as quickly as possible, but to adopt it responsibly and inclusively. With the right balance of innovation, human skills, regulation, and education, India can transform the AI revolution into an opportunity to build a more productive, inclusive, and technologically advanced nation.
"""

In [17]:
prompt = f"Evaluate the language quality of the following essay and provide detailed feedback and assign a score out of 10. Essay: {essay}"
structured_model.invoke(prompt).score

8

In [36]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    avg_score: float
    

In [37]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the language quality of the following essay and provide detailed feedback and assign a score out of 10. Essay: {state['essay']}"
    result = structured_model.invoke(prompt)

    return {'language_feedback': result.feedback, 'individual_scores': [result.score]}

In [38]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the analytical quality of the following essay and provide detailed feedback and assign a score out of 10. Essay: {state['essay']}"
    result = structured_model.invoke(prompt)

    return {'analysis_feedback': result.feedback, 'individual_scores': [result.score]}

In [39]:
def evaluate_clarity(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the clarity of the following essay and provide detailed feedback and assign a score out of 10. Essay: {state['essay']}"
    result = structured_model.invoke(prompt)

    return {'clarity_feedback': result.feedback, 'individual_scores': [result.score]}

In [40]:
def final_feedback(state: UPSCState) -> UPSCState:

    #summary feedback
    prompt = f"Provide a summary feedback for the following essay based on the language, analysis, and clarity feedback. Language Feedback: {state['language_feedback']}. Analysis Feedback: {state['analysis_feedback']}. Clarity Feedback: {state['clarity_feedback']}."
    overall_feedback = model.invoke(prompt).content

    #average calc
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [41]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_feedback', final_feedback)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_feedback')
graph.add_edge('evaluate_analysis', 'final_feedback')
graph.add_edge('evaluate_clarity', 'final_feedback')

graph.add_edge('final_feedback', END)

workflow = graph.compile()

In [42]:
initial_state = {'essay':essay}
workflow.invoke(initial_state)

{'essay': "# India in the Age of Artificial Intelligence\n\nArtificial Intelligence, commonly known as AI, is one of the most transformative technologies of the modern era. From smartphones and online shopping to healthcare, education, agriculture, banking, and manufacturing, AI is rapidly becoming part of everyday life. For a country as large and diverse as India, the age of AI presents both an enormous opportunity and a significant responsibility. India has the potential to become one of the world's leading AI-powered economies if it combines technological innovation with education, responsible development, and inclusive growth.\n\nIndia has several advantages in the AI revolution. The country has a large population of young people, a strong information-technology industry, a growing startup ecosystem, and a large pool of engineers and software professionals. Indian companies and startups are increasingly using AI to automate processes, analyze large amounts of data, improve customer